# Chapter 2: Regression

**Mastering Causal Metrics: An AI-Powered Study Guide**

*A companion to Mastering 'Metrics by Angrist & Pischke*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/02-regression.ipynb)

---

> 💡 **Learning Objectives**
>

By the end of this chapter, you will be able to:

- Explain how **regression controls** approximate experimental comparisons
- Write and interpret a **regression model** with treatment and control variables
- State the **Omitted Variables Bias (OVB) formula** and use it to predict the direction of bias
- Distinguish between **short** and **long** regressions
- Understand when adding controls helps --- and when it can make things worse (**bad controls**)
- Apply regression sensitivity analysis to assess the robustness of causal estimates


This chapter introduces regression --- the most widely used tool in the econometrician's toolkit. When randomized experiments are not available, regression lets us approximate an experimental comparison by holding observable characteristics constant.

> 📊 **Roadmap for Chapter 2** *(diagram — view in the [online book](https://github.com/cmg777/intro2causal))*

## Is a Private College Worth It?

Students at elite private universities in the United States pay roughly $20,000 more per year in tuition than those at public universities. Graduates of Harvard, Stanford, and Yale earn substantially more than graduates of state schools. But does the private school *cause* higher earnings, or are the students who attend these schools simply different --- smarter, more motivated, better connected --- in ways that would lead to high earnings regardless?

This is the same selection bias problem we met in Chapter 1. But here, we can't run a randomized experiment (Harvard's admissions office won't flip a coin). Instead, we reach for regression.

> 📝 **Intuition Builder: Regression as Automated Matching**
>

Think of regression as a matchmaking service. It finds pairs of students who look similar on paper --- same test scores, same family income, same types of schools applied to --- but one went private and the other went public. The regression estimate is like averaging the earnings difference across all these matched pairs.

When the matching is on **all the right variables**, regression approximates what a randomized experiment would show. When important variables are missing, the match is imperfect, and bias creeps in.

To separate the school's causal effect from the student's pre-existing advantages, we need a tool that holds observable characteristics constant. That tool is regression.



## How Regression Works

### The Regression Model

A regression links an outcome ($Y_i$) to a treatment variable ($P_i$) while holding control variables ($X_i$) constant:

$$Y_i = \alpha + \beta P_i + \gamma X_i + e_i$$

where:

- $\alpha$ = intercept (average outcome when $P_i = 0$ and $X_i = 0$)
- $\beta$ = the treatment effect we're after (how much $Y$ changes when $P$ switches from 0 to 1, holding $X$ constant)
- $\gamma$ = effect of the control variable
- $e_i$ = residual (everything else affecting $Y$ that's not in the model)

**OLS (Ordinary Least Squares)** chooses $\alpha$, $\beta$, and $\gamma$ to minimize the sum of squared residuals --- making the model's predictions as close to the actual data as possible.

> 📝 **Connection to Chapter 1**
>

In Chapter 1, we regressed outcomes on a treatment dummy with no controls. The coefficient was the difference in means between treated and untreated. Adding controls is the key innovation of Chapter 2: regression holds the controls constant, producing an "other things equal" comparison within groups that share the same control values.



## Seeing OVB with Simulated Data

To understand omitted variables bias, let's create a dataset where we **know the truth** --- because we designed it ourselves. This makes it easy to see when regression gets it right and when it goes wrong.

### The Data-Generating Process

We simulate 1,000 students choosing between private and public colleges:

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Set seed so everyone gets the same random numbers
np.random.seed(42)
n = 1000  # number of simulated students

# --- Step 1: Generate ABILITY (the unobserved confounder) ---
# Each student gets a random ability score (mean=50, sd=10)
ability = np.random.normal(50, 10, n)

# --- Step 2: Private school CHOICE depends on ability ---
# Higher ability → higher probability of choosing private school (logistic function)
# Students with ability above 50 have >50% chance; below 50 have <50% chance
prob_private = 1 / (1 + np.exp(-(ability - 50) / 5))
# Flip a coin for each student using their personal probability
private = np.random.binomial(1, prob_private)

# --- Step 3: EARNINGS depend on both private school AND ability ---
# The TRUE causal effect of private school is exactly $5,000
true_effect = 5000
# Base pay ($30,000) + private school bonus + ability bonus + random noise
earnings = (30000
            + true_effect * private
            + 800 * ability
            + np.random.normal(0, 5000, n))

# --- Step 4: Combine into a clean dataset ---
students = pd.DataFrame({
    "earnings": earnings,
    "private": private,
    "ability": ability,
})

students.head(5)

> ⭐ **The Ground Truth**
>

We built this data so that:

- The **true causal effect** of private school is exactly **$5,000**
- **Ability** independently increases earnings AND makes private school more likely
- This creates **selection bias**: private school students earn more partly because they're higher-ability, not just because of the school



### The Short Regression (Omitting Ability)

What happens if we regress earnings on `private` without controlling for ability?

In [ ]:
# SHORT regression: omit the confounder (ability)
short_model = smf.ols("earnings ~ private", data=students)
short = short_model.fit()

# Extract key regression results into a clear table
pd.DataFrame({
    "Variable": short.params.index,
    "Coefficient": short.params.round(2).values,
    "Std. Error": short.bse.round(2).values,
    "t-statistic": short.tvalues.round(2).values,
    "p-value": short.pvalues.round(3).values,
})

The coefficient on `private` is well above $5,000. This is **omitted variables bias** --- the regression attributes some of ability's effect to the private school dummy because the two are correlated.


### The Long Regression (Including Ability)

Now add ability as a control:

In [ ]:
# LONG regression: include the confounder (ability)
long_model = smf.ols("earnings ~ private + ability", data=students)
long = long_model.fit()

# Extract key regression results into a clear table
pd.DataFrame({
    "Variable": long.params.index,
    "Coefficient": long.params.round(2).values,
    "Std. Error": long.bse.round(2).values,
    "t-statistic": long.tvalues.round(2).values,
    "p-value": long.pvalues.round(3).values,
})

With ability controlled, the private school coefficient drops to approximately $5,000 --- close to the true causal effect we built into the data.

> ⚠️ **Common Misconception: "Just add more controls"**
>

Adding controls helps *only* when the controls are confounders (variables that affect both treatment and outcome). Adding irrelevant variables wastes statistical precision. And adding **bad controls** --- variables that are *caused by* the treatment --- can actually introduce bias. We return to this danger in Chapter 6.



## The OVB Formula

### The Most Important Equation in Econometrics

The relationship between the short and long regression coefficients follows a precise formula:

$$\text{OVB} = \beta^s - \beta^l = \underbrace{\pi_1}_{\text{Relationship between}\atop\text{omitted and treatment}} \times \underbrace{\gamma}_{\text{Effect of omitted}\atop\text{in long regression}}$$

where:

- $\beta^s$ = coefficient on treatment in the **short** regression (fewer controls)
- $\beta^l$ = coefficient on treatment in the **long** regression (more controls)
- $\pi_1$ = coefficient from regressing the **omitted variable** on the **treatment variable**
- $\gamma$ = coefficient on the **omitted variable** in the long regression

> 📝 **Intuition Builder: The Missing Ingredient**
>

Think of baking a cake. The recipe calls for flour, sugar, and eggs. If you forget the sugar (omitted variable), the cake will taste different from what you intended. The OVB formula tells you *how much* the taste changes and *in what direction*:

- **$\pi_1$**: How correlated is sugar with the other ingredients you *did* include? (If you always add sugar when you add flour, omitting sugar distorts the flour effect.)
- **$\gamma$**: How much does sugar matter for the final taste? (If sugar is critical, omitting it causes big bias.)
- **OVB = $\pi_1 \times \gamma$**: The bias is the product of these two factors.

If either factor is zero --- the omitted variable is unrelated to treatment, or it doesn't affect the outcome --- there's no bias.


### Verifying the OVB Formula

Let's check that the formula works with our simulated data:

In [ ]:
# --- Step 1: Get the short and long coefficients on "private" ---
beta_short = short.params["private"]
beta_long = long.params["private"]

# --- Step 2: Compute OVB directly (short minus long) ---
ovb_direct = beta_short - beta_long

# --- Step 3: Compute the two components of the OVB formula ---
# pi_1: regress the OMITTED variable (ability) on the TREATMENT (private)
aux_model = smf.ols("ability ~ private", data=students)
auxiliary = aux_model.fit()
pi_1 = auxiliary.params["private"]  # how much ability differs by private status

# gamma: coefficient on ability in the LONG regression
gamma = long.params["ability"]  # how much ability affects earnings

# --- Step 4: OVB from the formula (should match Step 2) ---
ovb_formula = pi_1 * gamma

# --- Display results ---
pd.DataFrame({
    "Component": [
        "Short reg coefficient (private)",
        "Long reg coefficient (private)",
        "OVB (direct: short - long)",
        "pi_1 (ability ~ private)",
        "gamma (ability in long reg)",
        "OVB (formula: pi_1 x gamma)",
    ],
    "Value": [
        round(beta_short),
        round(beta_long),
        round(ovb_direct),
        round(pi_1, 2),
        round(gamma),
        round(ovb_formula),
    ],
})

The formula matches. The two components reveal *why* the bias exists:

- **$\pi_1 > 0$**: Higher-ability students are more likely to attend private school
- **$\gamma > 0$**: Higher ability increases earnings
- **OVB = positive $\times$ positive = positive**: The short regression overstates the private school effect

### Predicting the Direction of Bias

Even when we can't observe the omitted variable, the OVB formula lets us **predict the direction of bias** by reasoning about the signs of $\pi_1$ and $\gamma$:

| $\pi_1$ (omitted ↔ treatment) | $\gamma$ (omitted → outcome) | OVB direction |
|:---:|:---:|:---:|
| Positive | Positive | **Upward** bias |
| Positive | Negative | **Downward** bias |
| Negative | Positive | **Downward** bias |
| Negative | Negative | **Upward** bias |

: The sign of OVB depends on the signs of both components {.striped}


## Case Study: The Private College Premium

### Dale and Krueger's Self-Revelation Model

Economists Stacy Dale and Alan Krueger studied the earnings of over 14,000 college students using the **College and Beyond (C&B)** dataset. Their key insight was that the schools students *applied to* reveal information about their ambition and ability.

**The matching strategy**: Compare students who were admitted to the same set of schools but chose to attend different ones. For example, a student admitted to both Harvard and UMass who chose Harvard versus one who chose UMass. Both students were *equally qualified* (admitted to the same schools), but made different enrollment decisions.

**The findings** (paraphrased):

- Without controls, private school graduates earned about **14% more** than public school graduates
- Controlling for Barron's selectivity group reduced this to about **7%**
- Controlling for the specific schools applied to (the "self-revelation" model) reduced it to **close to zero**

> ⭐ **Key Finding: The Private School Premium is Mostly Selection**
>

Once you compare students who were equally ambitious (applied to similar schools), the earnings advantage of attending an elite private college **largely disappears**. Most of the raw earnings gap reflects who attends private school, not what private school does.

This is a textbook demonstration of OVB at work: when you add the right controls, the treatment effect shrinks dramatically.

*Note: The C&B dataset is not publicly available, so we discuss Dale and Krueger's findings rather than replicating the analysis in code. The simulated data above demonstrated the same OVB principles that their study applies to real data.*


### Regression Sensitivity Analysis

The Dale and Krueger results illustrate an important robustness check: **sensitivity analysis**. When adding controls doesn't change the estimate much, we can be more confident that the remaining estimate isn't driven by further omitted variables.

In their data:

- Adding SAT scores, parental income, and demographics **barely changed** the private school coefficient once the self-revelation controls were included
- The OVB formula explains why: conditional on application behavior, private school attendance was **no longer correlated** with these variables ($\pi_1 \approx 0$), so omitting them caused little bias

The Dale and Krueger study succeeded because they controlled for the *right* variables --- pre-treatment characteristics like application behavior. But what happens when researchers control for the *wrong* variables?


## When Controls Go Wrong: Bad Controls

> ⚠️ **Not All Controls Are Good Controls**
>

A **bad control** is a variable that is *caused by* the treatment. Controlling for it blocks the causal pathway and distorts the estimate.

**Example**: Suppose private school causes students to enter higher-paying occupations. If you control for occupation, you're asking "among people in the same job, do private school grads earn more?" This removes one of the main ways private school helps, leading you to underestimate the true effect.

**Rule of thumb**: Only control for variables determined *before* the treatment was assigned. Variables determined *after* treatment (occupation, graduate degree, industry) are potential outcomes, not confounders.


> 📝 **Connection to Chapter 6**
>

Chapter 6 revisits bad controls in the context of returns to schooling. Controlling for occupation when estimating the effect of education is a classic bad-control mistake. The lesson is the same: controls must be *pre-treatment* characteristics, not downstream outcomes.



## How Regression Connects to Every Other Chapter

Regression is not just a standalone method --- it is the **building block** for every other tool in this book:

| Chapter | How Regression Appears |
|:---|:---|
| **Ch 1 (RCTs)** | Difference in means *is* a regression on a treatment dummy |
| **Ch 3 (IV)** | First stage and reduced form are regressions; 2SLS uses predicted values from regression |
| **Ch 4 (RD)** | RD regression controls for a polynomial in the running variable |
| **Ch 5 (DD)** | DD is a regression with group and time fixed effects |
| **Ch 6 (Schooling)** | OLS regression is the baseline; twins FE is a differenced regression |

: Regression is the foundation of all five methods in the book {.striped}


## Historical Perspective: Galton and Yule

### Francis Galton and "Regression to the Mean"

The word "regression" comes from **Sir Francis Galton** (1886), who studied the heights of parents and children. He observed that very tall parents tend to have children who are tall but *less extreme* than their parents --- heights "regress toward the mean." Galton's finding was about a statistical regularity, not causation, but the mathematical tool he developed to describe it became the foundation of modern regression analysis.

### George Udny Yule and Social Statistics

**George Udny Yule** (1899) was among the first to apply regression to social policy questions. He studied the causes of changes in pauperism (poverty) in England, using regression to control for multiple factors simultaneously. Yule's work pioneered the use of regression with multiple control variables --- exactly the approach we've been learning.

Both Galton and Yule worked in an era before causal inference was formalized. Their statistical tools were designed for description and prediction. The causal interpretation of regression --- asking whether $\beta$ represents a causal effect --- is a modern contribution that depends on the assumptions we've discussed (correct controls, no omitted variables).


## Key Takeaways

> 📊 **How the key concepts of Chapter 2 connect** *(diagram — view in the [online book](https://github.com/cmg777/intro2causal))*

1. **Regression approximates an experiment** by comparing treated and untreated observations that share the same values of control variables.

2. **OVB = $\pi_1 \times \gamma$** --- the bias from omitting a variable equals the correlation of the omitted variable with treatment times its effect on the outcome.

3. **The direction of OVB can be predicted** by reasoning about the signs of $\pi_1$ and $\gamma$, even when the omitted variable is unobserved.

4. **Sensitivity analysis**: If adding controls doesn't change the estimate much, we gain confidence that remaining omitted variables aren't causing large bias.

5. **Bad controls** (post-treatment variables) should never be included --- they block causal pathways and introduce new bias.

6. **Regression is foundational**: Every method in the book (IV, RD, DD) uses regression as a building block.

7. **The private college premium** largely disappears once you match students by the schools they applied to --- most of the raw gap is selection, not causation.


## Learn by Coding

Copy this code into a Python notebook to reproduce the key results from this chapter.

In [ ]:
# ============================================================
# Chapter 2: Regression — Code Cheatsheet
# ============================================================
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# --- Step 1: Simulate data where we KNOW the true causal effect ---
np.random.seed(42)
n = 1000
ability = np.random.normal(50, 10, n)
prob_private = 1 / (1 + np.exp(-(ability - 50) / 5))
private = np.random.binomial(1, prob_private)
true_effect = 5000
earnings = 30000 + true_effect * private + 800 * ability + np.random.normal(0, 5000, n)
students = pd.DataFrame({"earnings": earnings, "private": private, "ability": ability})

# --- Step 2: Short regression (omitting ability → biased) ---
short = smf.ols("earnings ~ private", data=students).fit()
print("SHORT regression (biased — omits ability):")
print(f"  Private school coefficient: ${short.params['private']:,.0f}")
print(f"  True effect is $5,000 — the estimate is too high!\n")

# --- Step 3: Long regression (including ability → unbiased) ---
long = smf.ols("earnings ~ private + ability", data=students).fit()
print("LONG regression (controls for ability):")
print(f"  Private school coefficient: ${long.params['private']:,.0f}")
print(f"  Close to the true effect of $5,000\n")

# --- Step 4: Verify the OVB formula ---
ovb_direct = short.params["private"] - long.params["private"]
aux = smf.ols("ability ~ private", data=students).fit()
pi_1 = aux.params["private"]       # relationship: omitted ↔ treatment
gamma = long.params["ability"]      # effect of omitted in long regression
ovb_formula = pi_1 * gamma
print("OVB Formula Verification:")
print(f"  Direct OVB (short - long):  ${ovb_direct:,.0f}")
print(f"  Formula OVB (pi1 x gamma): ${ovb_formula:,.0f}")
print(f"  pi1 = {pi_1:.2f}, gamma = {gamma:.0f}")

## Solutions

### Conceptual Questions

**Q1.** **The training coefficient is biased upward because it absorbs the positive effect of the omitted experience variable.**

1. **Identify $\pi_1$:** The relationship between experience and training is positive, since experienced workers tend to receive more training.
2. **Identify $\gamma$:** The effect of experience on wages in the long regression is positive, since experience raises wages.
3. **Apply the OVB formula:** OVB = $\pi_1 \times \gamma$ = positive $\times$ positive = **positive**.
4. **Conclude:** The short regression overstates the true effect of training because it partly captures the wage-boosting effect of experience. This is a direct application of the OVB formula introduced in the chapter.

**Q2.** **Omitting family income biases the class size effect downward, making smaller classes look more beneficial than they truly are.**

1. **Compute OVB:** OVB = short $-$ long = $-5 - (-2) = -3$.
2. **Decompose the sign:** Family income is negatively correlated with class size (richer families choose smaller classes, so $\pi_1 < 0$) and positively correlated with test scores ($\gamma > 0$). The product $\pi_1 \times \gamma$ is negative.
3. **Interpret:** The negative OVB means the short regression exaggerates the class size penalty --- some of the apparent class size effect was really a family income effect. This illustrates the chapter's warning: the direction of OVB depends on the signs of both $\pi_1$ and $\gamma$.

**Q3.** **Body weight is a bad control because it is a downstream consequence of exercise, not a pre-treatment confounder.**

1. **Identify the causal pathway:** Exercise causes changes in body weight, so weight is a mediator on the path exercise $\rightarrow$ lower weight $\rightarrow$ better mental health.
2. **Explain the problem:** Controlling for weight blocks this pathway, absorbing part of the total effect of exercise. The regression would understate how much exercise improves mental health.
3. **State the rule:** As the chapter emphasizes, only control for variables determined *before* the treatment (pre-treatment covariates). Bad controls --- variables that are themselves affected by treatment --- introduce bias by removing part of the causal effect you are trying to measure.

**Q4.** **Study A is more credible because coefficient stability across specifications signals low omitted variable bias.**

1. **Compare the shifts:** Study A's estimate barely changes when controls are added ($-3$ to $-2.8$, a shift of $0.2$). Study B's estimate drops dramatically ($-8$ to $-2$, a shift of $6$).
2. **Apply OVB logic:** By the OVB formula, the large change in Study B means the added controls were highly correlated with both class size ($\pi_1$ is large) and test scores ($\gamma$ is large). The uncontrolled estimate was severely biased.
3. **Draw the conclusion:** Study A's stability suggests that omitted variables are less of a concern --- the short and long regressions tell a similar story. This coefficient-stability heuristic is a practical diagnostic from the chapter: when adding controls barely moves the estimate, we gain confidence that further omitted variables are unlikely to change it much either.

**Q5.** **Regression can only deliver causal estimates if there are no unobserved confounders --- a strong assumption that is unlikely to hold here.**

1. **State the assumption:** The regression estimate is causal only if age, income, and diet are the *only* confounders (the conditional independence assumption, or CIA).
2. **List plausible violations:** Unobserved factors could still bias the result: genetics (some people are naturally healthier AND more inclined to exercise), motivation, social support, or pre-existing health conditions. Each of these is correlated with both exercise and health, creating OVB.
3. **Connect to the broader course:** Without random assignment of exercise, we can never be sure we have controlled for everything. This fundamental limitation of regression motivates the methods in later chapters --- instrumental variables (Chapter 3), regression discontinuity (Chapter 4), and differences-in-differences (Chapter 5) --- which rely on research designs rather than exhaustive control lists.

### Research Tasks

**R1.**

In [ ]:
# --- Generate data with NO causal effect ---
np.random.seed(42)
ability2 = np.random.normal(50, 10, n)
prob2 = 1 / (1 + np.exp(-(ability2 - 50) / 5))  # ability drives private school selection
private2 = np.random.binomial(1, prob2)
earnings2 = 30000 + 0 * private2 + 800 * ability2 + np.random.normal(0, 5000, n)  # true effect = 0

students2 = pd.DataFrame({"earnings": earnings2, "private": private2, "ability": ability2})

# --- Run short vs long regressions ---
short2 = smf.ols("earnings ~ private", data=students2).fit()             # omits ability (biased)
long2 = smf.ols("earnings ~ private + ability", data=students2).fit()    # includes ability (correct)

pd.DataFrame({
    "Regression": ["Short (omit ability)", "Long (include ability)"],
    "Private coefficient": [round(short2.params["private"]), round(long2.params["private"])],
    "True effect": [0, 0],
})

Stata equivalent:

```stata
* --- Simulation with true effect = 0 ---
clear all
set more off
set seed 42
set obs 1000
gen ability = rnormal(50, 10)
gen prob_private = 1 / (1 + exp(-(ability - 50) / 5))
gen private = rbinomial(1, prob_private)
gen earnings = 30000 + 0 * private + 800 * ability + rnormal(0, 5000)
reg earnings private
reg earnings private ability
```

(1) **What the numbers show:** The short regression shows a positive coefficient even though the true effect is zero. The long regression correctly recovers approximately zero.

(2) **Why:** This is pure OVB in action --- higher-ability students select into private school ($\pi_1 > 0$) AND ability raises earnings ($\gamma > 0$). The short regression attributes ability's effect to private schooling because the omitted variable is correlated with both the treatment and the outcome.

(3) **What it teaches:** OVB can create the illusion of a causal effect where none exists. This is the most dangerous form of bias: a policy maker relying on the short regression would incorrectly conclude that private schooling boosts earnings. The long regression, by controlling for the confounder, eliminates the bias.

**R2.**

In [ ]:
# --- Generate data with stronger confounder ---
np.random.seed(42)
ability3 = np.random.normal(50, 10, n)
prob3 = 1 / (1 + np.exp(-(ability3 - 50) / 2))  # divide by 2 instead of 5 = stronger selection
private3 = np.random.binomial(1, prob3)
earnings3 = 30000 + 5000 * private3 + 800 * ability3 + np.random.normal(0, 5000, n)
students3 = pd.DataFrame({"earnings": earnings3, "private": private3, "ability": ability3})

# --- Run short, long, and auxiliary regressions ---
short3 = smf.ols("earnings ~ private", data=students3).fit()       # biased estimate
long3 = smf.ols("earnings ~ private + ability", data=students3).fit()  # closer to true effect
aux3 = smf.ols("ability ~ private", data=students3).fit()          # estimates pi_1

# --- Verify OVB formula ---
ovb3 = round(short3.params["private"] - long3.params["private"])   # direct: short minus long
formula3 = round(aux3.params["private"] * long3.params["ability"]) # formula: pi_1 * gamma

pd.DataFrame({
    "Metric": ["Short coef", "Long coef", "OVB (direct)", "pi_1", "gamma", "OVB (formula)"],
    "Value": [round(short3.params["private"]), round(long3.params["private"]),
              ovb3, round(aux3.params["private"], 2), round(long3.params["ability"]),
              formula3],
})

Stata equivalent:

```stata
* --- Stronger confounder (divide by 2 instead of 5) ---
clear all
set more off
set seed 42
set obs 1000
gen ability = rnormal(50, 10)
gen prob_private = 1 / (1 + exp(-(ability - 50) / 2))
gen private = rbinomial(1, prob_private)
gen earnings = 30000 + 5000 * private + 800 * ability + rnormal(0, 5000)
reg earnings private
scalar short_coef = _b[private]
reg earnings private ability
scalar long_coef = _b[private]
scalar gamma = _b[ability]
reg ability private
scalar pi1 = _b[private]
scalar ovb_direct = short_coef - long_coef
scalar ovb_formula = pi1 * gamma
display "OVB (direct) = " ovb_direct
display "OVB (formula) = " ovb_formula
```

(1) **What the numbers show:** With a stronger ability-private link, $\pi_1$ increases substantially and the OVB grows. The short regression coefficient is now much further from the true effect of 5,000. The OVB formula ($\pi_1 \times \gamma$) matches the direct calculation (short $-$ long), confirming the formula works exactly.

(2) **Why:** A tighter link between ability and private school selection (dividing by 2 instead of 5 in the logistic function) means ability is a stronger predictor of treatment. Since $\gamma$ (the effect of ability on earnings) stays the same, the larger $\pi_1$ mechanically produces larger OVB.

(3) **What it teaches:** The magnitude of OVB depends on how strongly the omitted variable predicts treatment ($\pi_1$). This is a key practical insight from the chapter: even if you know the direction of bias, the size of the problem depends on how strongly confounders sort people into treatment and control groups.

**R3.**

In [ ]:
# --- Generate data with two confounders ---
np.random.seed(42)
ability4 = np.random.normal(50, 10, n)
family_income = np.random.normal(60000, 20000, n)
# Both ability and income affect private school choice
prob4 = 1 / (1 + np.exp(-((ability4 - 50) / 5 + (family_income - 60000) / 20000)))
private4 = np.random.binomial(1, prob4)
# Both ability and income affect earnings
earnings4 = 10000 + 5000 * private4 + 800 * ability4 + 0.3 * family_income + np.random.normal(0, 5000, n)

students4 = pd.DataFrame({
    "earnings": earnings4, "private": private4,
    "ability": ability4, "family_income": family_income,
})

# --- Run three regressions with progressively more controls ---
r_short = smf.ols("earnings ~ private", data=students4).fit()                        # no controls
r_medium = smf.ols("earnings ~ private + ability", data=students4).fit()             # one control
r_long = smf.ols("earnings ~ private + ability + family_income", data=students4).fit()  # both controls

pd.DataFrame({
    "Regression": ["Short (no controls)", "Medium (ability only)", "Long (ability + income)"],
    "Private coefficient": [round(r_short.params["private"]),
                             round(r_medium.params["private"]),
                             round(r_long.params["private"])],
    "True effect": [5000, 5000, 5000],
})

Stata equivalent:

```stata
* --- Two confounders: ability and family income ---
clear all
set more off
set seed 42
set obs 1000
gen ability = rnormal(50, 10)
gen family_income = rnormal(60000, 20000)
gen prob_private = 1 / (1 + exp(-((ability - 50) / 5 + (family_income - 60000) / 20000)))
gen private = rbinomial(1, prob_private)
gen earnings = 10000 + 5000 * private + 800 * ability + 0.3 * family_income + rnormal(0, 5000)
reg earnings private
reg earnings private ability
reg earnings private ability family_income
```

(1) **What the numbers show:** The short regression (no controls) is the most biased. Adding ability alone (medium) moves the coefficient closer to the true 5,000, but it still overshoots. Adding both ability and family income (long) gets closest to the true effect.

(2) **Why:** Each omitted confounder contributes its own OVB term. Family income is positively correlated with both private school attendance and earnings, so omitting it inflates the private school coefficient. Adding ability removes one source of bias but leaves the income-driven bias in place.

(3) **What it teaches:** With multiple confounders, controlling for only some of them reduces bias but does not eliminate it. The progression from short to medium to long regression illustrates the chapter's core message: the long regression moves toward the causal effect only when it includes *all* relevant confounders. In practice, we can never be certain we have controlled for everything --- which is why the book introduces stronger research designs in later chapters.

**R4.**

In [ ]:
# --- Generate data with reversed confounder ---
np.random.seed(42)
ability_r = np.random.normal(50, 10, n)
prob_r = 1 / (1 + np.exp((ability_r - 50) / 5))
private_r = np.random.binomial(1, prob_r)
earnings_r = 30000 + 5000 * private_r + 800 * ability_r + np.random.normal(0, 5000, n)
students_r = pd.DataFrame({"earnings": earnings_r, "private": private_r, "ability": ability_r})

short_r = smf.ols("earnings ~ private", data=students_r).fit()
long_r = smf.ols("earnings ~ private + ability", data=students_r).fit()
aux_r = smf.ols("ability ~ private", data=students_r).fit()

ovb_r = round(short_r.params["private"] - long_r.params["private"])
formula_r = round(aux_r.params["private"] * long_r.params["ability"])

pd.DataFrame({
    "Metric": ["Short coef", "Long coef", "OVB (direct)", "pi_1", "gamma", "OVB (formula)"],
    "Value": [round(short_r.params["private"]), round(long_r.params["private"]),
              ovb_r, round(aux_r.params["private"], 2), round(long_r.params["ability"]), formula_r],
})

Stata equivalent:

```stata
* --- Reversed confounder ---
clear all
set more off
set seed 42
set obs 1000
gen ability = rnormal(50, 10)
gen prob_private = 1 / (1 + exp((ability - 50) / 5))
gen private = rbinomial(1, prob_private)
gen earnings = 30000 + 5000 * private + 800 * ability + rnormal(0, 5000)
reg earnings private
scalar short_coef = _b[private]
reg earnings private ability
scalar long_coef = _b[private]
scalar gamma = _b[ability]
reg ability private
scalar pi1 = _b[private]
display "OVB (direct) = " short_coef - long_coef " (should be negative)"
display "OVB (formula) = " pi1 * gamma
```

(1) **What the numbers show:** The short regression coefficient is now *below* the true effect of 5,000. The OVB is negative.

(2) **Why:** When high-ability students attend strong public schools ($\pi_1 < 0$), the private school group has lower average ability.

(3) **What it teaches:** The OVB formula works for all four sign combinations. Downward bias occurs when treatment is negatively selected.

**R5.**

In [ ]:
# --- Generate data with three confounders ---
np.random.seed(42)
ability_s = np.random.normal(50, 10, n)
income_s = np.random.normal(60000, 20000, n)
motivation = np.random.normal(5, 2, n)
prob_s = 1 / (1 + np.exp(-((ability_s - 50)/5 + (income_s - 60000)/20000 + (motivation - 5)/2)))
private_s = np.random.binomial(1, prob_s)
earnings_s = (10000 + 5000 * private_s + 800 * ability_s + 0.3 * income_s + 2000 * motivation + np.random.normal(0, 5000, n))
df_s = pd.DataFrame({"earnings": earnings_s, "private": private_s, "ability": ability_s, "income": income_s, "motivation": motivation})

specs = [("No controls", "earnings ~ private"), ("+ ability", "earnings ~ private + ability"),
         ("+ ability + income", "earnings ~ private + ability + income"),
         ("+ all three", "earnings ~ private + ability + income + motivation")]
rows = []
for label, formula in specs:
    r = smf.ols(formula, data=df_s).fit()
    rows.append({"Specification": label, "Private coefficient": round(r.params["private"]), "True effect": 5000})
pd.DataFrame(rows)

Stata equivalent:

```stata
* --- Progressive control addition ---
clear all
set more off
set seed 42
set obs 1000
gen ability = rnormal(50, 10)
gen income = rnormal(60000, 20000)
gen motivation = rnormal(5, 2)
gen prob_private = 1 / (1 + exp(-((ability - 50)/5 + (income - 60000)/20000 + (motivation - 5)/2)))
gen private = rbinomial(1, prob_private)
gen earnings = 10000 + 5000*private + 800*ability + 0.3*income + 2000*motivation + rnormal(0, 5000)
reg earnings private
reg earnings private ability
reg earnings private ability income
reg earnings private ability income motivation
```

(1) **What the numbers show:** The coefficient starts far from 5,000 and converges as controls are added.

(2) **Why:** Each omitted confounder contributes its own OVB term. Adding all three approaches the true causal effect.

(3) **What it teaches:** This is the coefficient stability diagnostic: if adding controls barely changes the estimate, remaining OVB is likely small.